# Probe 047 launcher (phase B)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**Setup:** connect your Google Drive. No GitHub write token is needed.

Contract-bound source label: `results/probe-047-dc586665d0be` (no automatic push)

Per session: run all cells top to bottom. After a disconnect, rerun all cells. Resumable runners retain checkpoints; empty-directory runners refuse and require a new OUTPUT_DIR. Return the verified export and sibling console for review.

In [ ]:
PHASE = 'B'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = '194cdcd491a608190fca45c24ad216153431270e'
RESULTS_BRANCH = 'results/probe-047-dc586665d0be'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/047_B_dc586665d0be'

In [ ]:
from google.colab import drive, userdata
import os, shutil, time
MP = '/content/drive'
def _mounted(mp):
    try:
        return any(len(l.split()) > 1 and l.split()[1] == mp
                   for l in open('/proc/mounts'))
    except OSError:
        return False
try:
    if os.path.isdir(MP) and not _mounted(MP) and os.listdir(MP):
        stale = f'/content/drive_stale_{int(time.time())}'
        shutil.move(MP, stale)
        print('stale mountpoint residue moved to', stale,
              '(a crashed FUSE mount left a corpse on a surviving VM)')
except OSError as e:
    print('could not inspect/move mountpoint residue:', e,
          '-- if the mount below fails, Runtime > Disconnect and delete runtime')
drive.mount(MP, force_remount=True)
# Optional model-download credential; this exporter needs no write token.
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    pass

In [ ]:
%cd /content
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

In [ ]:
!pip install -q -r probes/047/requirements.txt

In [ ]:
# --- origin_direct staging: the pinned Zenodo record is the source;
# the Drive mount carries ONLY small inputs/outputs (FUSE is out of
# the 99 GB path entirely -- seven recorded casualties).
import os, json, urllib.request
LOCAL = '/content/work'
os.makedirs(LOCAL, exist_ok=True)
RECORD_JSON = LOCAL + '/zenodo_record.json'
with urllib.request.urlopen('https://zenodo.org/api/records/16813698') as r:
    rec = json.load(r)
assert str(rec['id']) == '16813698', 'server returned a different record than the declared pin'
json.dump(rec, open(RECORD_JSON, 'w'), indent=2)
_a = [f for f in rec['files'] if f['key'].endswith('.7z')]
assert len(_a) == 1, _a
ARCHIVE_LOCAL = LOCAL + '/' + _a[0]['key']
ARCHIVE_URL = _a[0]['links']['self']
print('pinned record', rec['id'], _a[0]['key'], round(_a[0]['size']/1e9, 1), 'GB (origin_direct)')

In [ ]:
import hashlib, subprocess
_name = os.path.basename(ARCHIVE_LOCAL)
_entries = [f for f in rec['files'] if f.get('key') == _name]
assert len(_entries) == 1
_ck = _entries[0].get('checksum', '')
if not _ck.startswith('md5:'):
    raise SystemExit('driver configuration error: pinned record supplies no '
                     'md5 for ' + _name + '; refusing a 99 GB staging pass')
EXPECT_MD5 = _ck.split(':', 1)[1]
EXPECT_SIZE = _entries[0]['size']
def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()
subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=False)
LOCAL_DATA = LOCAL + '/extracted'
_done = False
if os.path.exists(ARCHIVE_LOCAL):
    print('verifying existing local archive md5 (~4 min)...')
    if os.path.getsize(ARCHIVE_LOCAL) == EXPECT_SIZE and _md5(ARCHIVE_LOCAL) == EXPECT_MD5:
        _done = True
    else:
        print('existing local archive fails integrity; removing')
        os.remove(ARCHIVE_LOCAL)
if not _done:
    for _attempt in (1, 2):
        _part = _name + '.part'
        _nx = '16' if _attempt == 1 else '4'
        _lg = LOCAL + '/aria2_attempt' + str(_attempt) + '.log'
        print('downloading from Zenodo (attempt', _attempt,
              'of 2;', _nx + '-way aria2c;',
              round(EXPECT_SIZE/1e9, 1), 'GB -- ETA prints below)...')
        _rc = subprocess.run(['aria2c', '-x' + _nx, '-s' + _nx, '-k4M',
                              '--file-allocation=none', '-c',
                              '--log', _lg, '--log-level=notice',
                              '-d', LOCAL, '-o', _part, ARCHIVE_URL]).returncode
        _pp = LOCAL + '/' + _part
        if (_rc == 0 and os.path.exists(_pp)
                and os.path.getsize(_pp) == EXPECT_SIZE):
            print('verifying downloaded bytes md5 (~4 min)...')
            if _md5(_pp) == EXPECT_MD5:
                os.replace(_pp, ARCHIVE_LOCAL)
                _done = True
                break
        _full = os.path.exists(_pp) and os.path.getsize(_pp) == EXPECT_SIZE
        if _rc == 0 and _full:
            print('full-size download failed md5; discarding poisoned bytes')
            for _f in (_pp, _pp + '.aria2'):
                if os.path.exists(_f):
                    os.remove(_f)
        else:
            print('transfer interrupted (rc', _rc, '); KEEPING partial for resume')
        if os.path.exists(_lg):
            print('aria2 log tail:')
            print(open(_lg, errors='replace').read()[-700:])
if not _done:
    raise SystemExit('ORIGIN_DOWNLOAD_INTEGRITY_FAILURE: two direct downloads '
                     'from the pinned record failed; check Zenodo status and '
                     'network, then rerun')
print('local archive verified: md5', EXPECT_MD5)
print('local archive bytes:', os.path.getsize(ARCHIVE_LOCAL))

In [ ]:
SUFFIXES = ['.nomatch']
if not os.path.isdir(LOCAL_DATA):
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'p7zip-full'], check=False)
    _rc = subprocess.run(['7z', 'x', ARCHIVE_LOCAL, '-o' + LOCAL_DATA, '-y']
                         + ['-ir!*' + x for x in SUFFIXES],
                         stdout=subprocess.DEVNULL).returncode
    assert _rc == 0, f'7z extraction failed rc={_rc} -- refusing to proceed'
_n = sum(len(f) for _, _, f in os.walk(LOCAL_DATA))
print('extracted files (local):', _n)
assert _n >= 800, 'extraction incomplete -- refusing to reach the census'
def _gzip_sweep(root):
    bad = []
    for _dp, _, _fs in os.walk(root):
        for _f in _fs:
            if _f.endswith('.gz'):
                _p = os.path.join(_dp, _f)
                if subprocess.run(['gzip', '-t', _p], capture_output=True).returncode:
                    bad.append(_p)
    return bad
_bad = _gzip_sweep(LOCAL_DATA)
if _bad:
    print('integrity sweep:', len(_bad), 'truncated/corrupt member(s); re-extracting just those')
    for _p in _bad:
        os.remove(_p)
    _rc2 = subprocess.run(['7z', 'x', ARCHIVE_LOCAL, '-o' + LOCAL_DATA, '-y']
                          + ['-ir!*' + os.path.basename(_p) for _p in _bad],
                          stdout=subprocess.DEVNULL).returncode
    assert _rc2 == 0, f'7z re-extraction failed rc={_rc2}'
    _bad2 = _gzip_sweep(LOCAL_DATA)
    _src_defects = []
    for _p in list(_bad2):
        _t = subprocess.run(['7z', 't', ARCHIVE_LOCAL,
                             '-ir!*' + os.path.basename(_p)],
                            stdout=subprocess.DEVNULL,
                            stderr=subprocess.DEVNULL).returncode
        if _t == 0:
            print('[SOURCE_MEMBER_DEFECT]', os.path.basename(_p),
                  '-- gzip stream invalid inside the md5-verified archive;',
                  'leaving in place for contract-level handling')
            _src_defects.append(_p)
            _bad2.remove(_p)
    assert not _bad2, ('EXTRACTION_INTEGRITY_FAILURE: '
        + '; '.join(os.path.basename(_x) for _x in _bad2[:5]))
    print('integrity sweep:', len(_src_defects), 'source-defect member(s) tolerated')
else:
    print('integrity sweep: all extracted members pass gzip -t')

In [ ]:
# Preserve checkpoints. Empty-destination runners refuse nondestructively.
from orchestrator.publication import run_logged
import shlex
!apt-get -qq install -y p7zip-full > /dev/null
CONSOLE = OUTPUT_DIR.rstrip('/') + '.console.log'
command = f'python probes/047/run.py --output-dir {OUTPUT_DIR} --archive-file {ARCHIVE_LOCAL} --member-manifest probes/023/results/results_v2/archive_manifest.csv'
RUN_EXIT = run_logged(shlex.split(command), OUTPUT_DIR)
print('Runner exit:', RUN_EXIT, '; original console:', CONSOLE)


In [ ]:
# Publication is an explicit reviewed export; no automatic commit or push.
# Failure evidence remains on Drive even when export is refused.
import json, pathlib
from orchestrator.publication import export_session
policy_path = pathlib.Path('probes/047/publication.json')
if not policy_path.is_file():
    raise RuntimeError('No reviewed publication policy; retain local outputs')
policy = json.loads(policy_path.read_text())
assert policy['contract_blob'] == 'dc586665d0bece940d1a1f4b3b0572f8c951c2ba', 'Publication policy contract drift'
EXPORT_DIR = OUTPUT_DIR.rstrip('/') + '.publication'
export_session(OUTPUT_DIR, EXPORT_DIR, policy)
print('Verified export:', EXPORT_DIR)
print('Return this export AND the sibling console for validation/import.')


Return the verified publication export and original sibling console. Validation and provenance-bound import precede interpretation. This launcher never commits or pushes results.